In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: LC DEMOGRAPHIC GROUP TERMS (LCDGT) INGESTION
# 
# ARCHITECTURE NOTE: LCDGT categorizes people into demographic groups using 
# broad thematic collections (e.g., Religion, Occupation). While most target 
# concepts live in the Religion collection, certain roles (like clergy) live 
# in the Occupation collection. 
#
# SSSOM ALIGNMENT STRATEGY: 
# This script uses a Hybrid Discovery approach:
# 1. It queries the LoC search API to harvest all URIs in the Religion collection.
# 2. It does a recursive downward crawl from explicit target seeds (like Clergy) 
#    to catch relevant occupational branches.
# 3. It uses a recursive bottom-up function to build the `Hierarchy_Path`, 
#    dynamically extracting the concept's MADS Collection to serve as the 
#    absolute root of the breadcrumb trail (e.g., "Occupation > Clergy").
# ==============================================================================

import os
import sys
import requests
import pandas as pd
import time
import re
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
load_dotenv("../../config/.env") 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
os.makedirs(raw_data_dir, exist_ok=True)

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory. Check PYTHONPATH.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "LCDGT"

registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="LC Demographic Group Terms",
    fallback_uri="http://id.loc.gov/authorities/demographicTerms/"
)

SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

# Harvest the entire Religion collection
COLLECTION_URL = "https://id.loc.gov/search/?q=memberOf:http://id.loc.gov/authorities/demographicTerms/collection_LCDGT_Religion&fo=json&count=500"

# Explicit seeds for branches outside the Religion collection
SEED_URIS = [
    "http://id.loc.gov/authorities/demographicTerms/dg2015060401", # Clergy (Occupation)
    "http://id.loc.gov/authorities/demographicTerms/dg2016060118", # Theologians
    "http://id.loc.gov/authorities/demographicTerms/dg2016060276", # Religion teachers
    "http://id.loc.gov/authorities/demographicTerms/dg2016060300", # Missionaries
    "http://id.loc.gov/authorities/demographicTerms/dg2015060206", # Nuns
    "http://id.loc.gov/authorities/demographicTerms/dg2017060113", # Islamicists
    "http://id.loc.gov/authorities/demographicTerms/dg2017060050", # Biblical scholars
    "http://id.loc.gov/authorities/demographicTerms/dg2024060191", # Hebraists
    "http://id.loc.gov/authorities/demographicTerms/dg2017060183", # Muftis
    "http://id.loc.gov/authorities/demographicTerms/dg2015060205" # Monks
]

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/json'
}

label_cache = {}
path_cache = {}
member_uris = set()

# --- 3. Persistent Progress & Schema Drift Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: Found {len(processed_ids)} LCDGT concepts already saved.")
else:
    processed_ids = set()
    print("[*] Starting fresh LCDGT ingest.")

# --- 4. Helper Functions (Ancestry & Extraction) ---
def get_label_only(uri):
    """Fetches just the English preferred label for collections or generic nodes."""
    if not uri: return ""
    clean_u = uri.replace('.json', '').replace('.rdf', '').replace('.html', '').rstrip('/')
    
    if clean_u in label_cache: return label_cache[clean_u]
    try:
        res = requests.get(f"{clean_u}.json", headers=HEADERS, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_u), {})
            
            for l_node in node.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
                lang = l_node.get('@language', '').lower()
                if not lang or lang.startswith('en'):
                    label = l_node.get('@value', '')
                    # Format collection names cleanly (e.g., "LCDGT Religion" -> "Religion")
                    if "LCDGT" in label: label = label.replace("LCDGT ", "").strip()
                    label_cache[clean_u] = label
                    return label
    except Exception: 
        pass
    return clean_u.split('/')[-1]

def get_full_lc_path(uri, depth=0):
    """Recursively fetches broader terms, appending a specific thematic MADS collection as the root."""
    if not uri: return ""
    clean_uri = uri.replace('.json', '').replace('.rdf', '').replace('.html', '').rstrip('/')
    
    if clean_uri in path_cache: return path_cache[clean_uri]
    if depth > 10: return clean_uri.split('/')[-1] # Safety cutoff
        
    try:
        res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
            
            # Extract Label
            label = clean_uri.split('/')[-1]
            for pref in node.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
                if pref.get('@language', 'en').startswith('en'): 
                    label = pref.get('@value', label)
                    label_cache[clean_uri] = label

            # Find Broader Parent
            broader_nodes = node.get('http://www.w3.org/2004/02/skos/core#broader', [])
            if not broader_nodes:
                broader_nodes = node.get('http://www.loc.gov/mads/rdf/v1#hasBroaderAuthority', [])
            
            # Recurse Upward OR Find Collection if at the root
            if broader_nodes and broader_nodes[0].get('@id'):
                parent_uri = broader_nodes[0].get('@id')
                parent_path = get_full_lc_path(parent_uri, depth + 1)
                full_path = f"{parent_path} > {label}" if parent_path else label
                path_cache[clean_uri] = full_path
                return full_path
            else:
                # We hit the top of the tree. Let's find its MADS collection.
                collection_nodes = node.get('http://www.loc.gov/mads/rdf/v1#isMemberOfMADSCollection', [])
                best_c_uri = None
                
                # Filter out the 'General' collection to prioritize thematic ones
                for c_node in collection_nodes:
                    c_uri = c_node.get('@id')
                    if c_uri and "collection_LCDGT_General" not in c_uri:
                        best_c_uri = c_uri
                        break
                
                # Fallback to General only if no other collection exists
                if not best_c_uri and collection_nodes and collection_nodes[0].get('@id'):
                    best_c_uri = collection_nodes[0].get('@id')
                
                if best_c_uri:
                    c_label = get_label_only(best_c_uri)
                    full_path = f"{c_label} > {label}" if c_label else label
                    path_cache[clean_uri] = full_path
                    return full_path
                else:
                    path_cache[clean_uri] = label
                    return label
    except Exception:
        pass
    
    return clean_uri.split('/')[-1]

def crawl_descendants(uri):
    """Recursively adds a seed and all its narrower children to the processing queue."""
    clean_uri = uri.replace('.json', '').replace('.rdf', '').replace('.html', '').rstrip('/')
    if clean_uri in member_uris: return
    
    member_uris.add(clean_uri)
    try:
        res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
            
            narrower_nodes = node.get('http://www.w3.org/2004/02/skos/core#narrower', [])
            for child in narrower_nodes:
                if child.get('@id'):
                    time.sleep(0.5) # Politeness
                    crawl_descendants(child.get('@id'))
    except Exception:
        pass

def get_lc_details(uri):
    """Fetches full metadata for an LC demographic term."""
    clean_uri = uri.replace('.json', '').replace('.rdf', '').replace('.html', '').rstrip('/')
    
    for attempt in range(3):
        try:
            res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=15)
            if res.status_code != 200: 
                time.sleep(1)
                continue
            
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
            
            SKOS_PREF = 'http://www.w3.org/2004/02/skos/core#prefLabel'
            SKOS_ALT  = 'http://www.w3.org/2004/02/skos/core#altLabel'
            SKOS_BROADER = 'http://www.w3.org/2004/02/skos/core#broader'
            MADS_CITATION = 'http://www.loc.gov/mads/rdf/v1#citationNote'

            label = "No Label"
            has_translation = ""
            synonyms = []
            
            for pref in node.get(SKOS_PREF, []):
                lang = pref.get('@language', '').lower()
                if not lang or lang.startswith('en'):
                    label = pref.get('@value', label)
                else:
                    has_translation = "yes"
            
            label_cache[clean_uri] = label
            
            for alt in node.get(SKOS_ALT, []):
                lang = alt.get('@language', '').lower()
                if not lang or lang.startswith('en'):
                    if alt.get('@value'): synonyms.append(alt.get('@value'))
                else:
                    has_translation = "yes"
            
            descriptions = []
            for item in data:
                if isinstance(item, dict):
                    for n in item.get(MADS_CITATION, []):
                        if n.get('@value'): descriptions.append(n.get('@value'))
            
            broader_nodes = node.get(SKOS_BROADER, [])
            p_ids = [b.get('@id').split('/')[-1] for b in broader_nodes if b.get('@id')]

            # Build absolute path, including the root collection
            absolute_path = get_full_lc_path(clean_uri)

            return {
                'label': label,
                'synonyms': " | ".join(list(set(synonyms))),
                'description': " | ".join(descriptions),
                'parent_ids': " | ".join(p_ids),
                'hierarchy_path': absolute_path, 
                'has_translation': has_translation
            }
        except Exception:
            time.sleep(2) 
    return None

# --- 5. Discovery Phase ---
print(f"Discovering Concepts for {SOURCE_PREFIX}...")

# 5a. Harvest Religion Collection
try:
    print(" -> Harvesting Religion Collection...")
    response = requests.get(COLLECTION_URL, headers=HEADERS, timeout=20)
    if response.status_code == 200:
        search_data = response.json()
        
        def extract_uris(obj):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if isinstance(v, str) and '/demographicTerms/dg' in v:
                        match = re.search(r'(dg\d+)', v)
                        if match: 
                            member_uris.add(f"{BASE_URI}{match.group(1)}")
                    extract_uris(v)
            elif isinstance(obj, list):
                for i in obj: 
                    extract_uris(i)

        extract_uris(search_data)
    else:
        print(f"Search failed. HTTP Status: {response.status_code}")
except Exception as e:
    print(f"Discovery error: {e}")

# 5b. Harvest Specific Seeds (e.g., Occupational Roles)
print(f" -> Harvesting explicit seeds and their descendants...")
for seed in SEED_URIS:
    crawl_descendants(seed)

print(f"Discovery complete. Found {len(member_uris)} distinct URIs to process.")

# --- 6. Extraction Phase ---
total = len(member_uris)
if total > 0:
    print(f"\nStarting ingest for {total} terms...")
    
    for i, uri in enumerate(sorted(list(member_uris)), 1):
        cid = uri.split('/')[-1]
        if cid in processed_ids: continue
            
        meta = get_lc_details(uri)
        if not meta: continue
            
        display_text = f"[{i}/{total}] Ingesting: {meta['label'][:60]}"
        print(f"\r{display_text}{' ' * 40}", end="", flush=True)

        extracted_data = {
            'Source_System': SOURCE_NAME, 
            'Primary_Label': meta['label'],
            'CURIE': f"{SOURCE_PREFIX}:{cid}", 
            'Formal_Label': meta['label'],
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': meta['hierarchy_path'], 
            'Synonyms': meta['synonyms'],
            'Description': meta['description'], 
            'Parent_IDs': meta['parent_ids'],
            'Concept_ID': cid, 
            'URI': uri, 
            'Has_Translation': meta['has_translation'],
            'Status': 'active',
            'Crosswalks': "" # LCDGT doesn't reliably contain external crosswalks
        }
        
        clean_row = finalize_row(extracted_data)
        pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
            raw_ingest_file, mode='a', index=False, 
            header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        
        processed_ids.add(cid)
        time.sleep(0.5) 
else:
    print("\nNo URIs found to ingest.")

print(f"\n\nIngest Complete! Saved {len(processed_ids)} records to {raw_ingest_file}")